In [2]:
!pip install airportsdata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.8/934.8 kB 13.9 MB/s eta 0:00:00


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import glob
from sklearn.preprocessing import LabelEncoder, StandardScaler, TargetEncoder
import airportsdata

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [3]:
import time
from sklearn.metrics import root_mean_squared_error as rmse
from sklearn.metrics import mean_absolute_error, r2_score

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [5]:
#We use a custom dataset due to the unique nature of our data. We train on sequences and a single flight, the next in the sequence, whose delay we would like to guess.
#Sequences_cont is all the fields we want to use for the sequence we are training on.
#Target_cont is all the fields we want to use for the single next slight in the sequence.
#The difference is that sequences_cont we use DepDelay and ArrDelay as features but Target_cont we don't.
#This is not cheating, since we are never using the future to predict the past.
class FlightDataset(Dataset):
    def __init__(self, sequences_cont, target_cont, labels):
        self.sequences_cont = torch.tensor(sequences_cont, dtype=torch.float32)
        self.target_cont = torch.tensor(target_cont, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.sequences_cont[idx],
            self.target_cont[idx],
            self.labels[idx]
        )

In [6]:
#Can play around with hidden_size (the number of nodes to train on in the LSTM), the dropout ratio, the structure of nn.classifier
class FlightDelayRNN(nn.Module):
    def __init__(
        self,
        n_cont_features,
        n_target_cont_features,
        dropout = 0.1,
        hidden_size =32
    ):
        super(FlightDelayRNN, self).__init__()

        self.cont_bn = nn.BatchNorm1d(n_cont_features) #batch normalization

        lstm_input_size = n_cont_features

        self.lstm = nn.LSTM(
            input_size=lstm_input_size,
            hidden_size=hidden_size,
            batch_first=True,
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + n_target_cont_features, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1)
        )

    def forward(self, x_cont, target_cont):
        output, (h_n, c_n) = self.lstm(x_cont)
        final_hidden = h_n[-1]
        observation = torch.cat([final_hidden, target_cont], dim=-1)
        return self.classifier(observation).squeeze(1)

In [7]:
def train_model(model, train_loader, val_loader):
    criterion = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=2, factor=0.5
    )

    best_val_loss = float('inf')

    for epoch in range(4):
        print(f'Beginning Epoch {epoch+1}')
        model.train()
        train_losses = []

        #training
        for  x_cont, x_target_cont, labels in train_loader:
            x_cont = x_cont.to(device)
            x_target_cont = x_target_cont.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            predictions = model( x_cont,  x_target_cont)
            loss = criterion(predictions, labels)
            loss.backward()

            # Gradient clipping — important for RNNs
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            train_losses.append(loss.item())


        model.eval()
        val_losses = []

        #validation
        with torch.no_grad():
            for x_cont, x_target_cont, labels in val_loader:
                x_cont = x_cont.to(device)
                x_target_cont = x_target_cont.to(device)
                labels = labels.to(device)

                predictions = model( x_cont, x_target_cont)
                loss = criterion(predictions, labels)
                val_losses.append(loss.item())

        avg_train_loss = np.mean(train_losses)
        avg_val_loss = np.mean(val_losses)

        scheduler.step(avg_val_loss)

        # save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_model.pt')

        print(f'Epoch {epoch+1:3d} | '
              f'Train Loss: {avg_train_loss:.4f} | '
              f'Val Loss: {avg_val_loss:.4f}')

    print('Training complete. Best model saved.')

In [6]:
#for the first ten months of 2023. you can change this around as well.
file_paths = glob.glob('/content/drive/MyDrive/Erdos Project Data/combined-data-fix/*.csv')
df = pd.concat([pd.read_csv(f) for f in file_paths[25:35]], ignore_index=True)

In [9]:
#cont_features - the features we use in our sequence
#target_cont_features - the features we use for the flight we want to predict the delay of
cont_features = ['Origin', 'Dest','Reporting_Airline', 'Hour_sin', 'Hour_cos', 'Month_sin','Month_cos', 'DayofMonth', 'delta_t', 'Year', 'CRSDepTime', 'CRSArrTime', 'dep_mx2t', 'dep_mn2t', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10',
                 'arr_mx2t', 'arr_mn2t', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10', 'DepDelay', 'ArrDelay']
target_cont_features = ['Origin', 'Dest', 'Reporting_Airline', 'Hour_sin','Hour_cos', 'Month_sin','Month_cos', 'DayofMonth', 'delta_t', 'Year', 'CRSDepTime', 'CRSArrTime', 'dep_mx2t', 'dep_mn2t', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10',
                 'arr_mx2t', 'arr_mn2t', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10']

In [7]:
from datetime import datetime, timezone
import zoneinfo

In [8]:
iata_airports = airportsdata.load('IATA')

In [9]:
# takes in a date and military time at a certain airport and returns the UTC time
# this function is no longer being used
def convert_date(year, month, day, hhmm, airport):
    dt = datetime.strptime(str(hhmm).zfill(4), '%H%M')
    dt = dt.replace(year = year, month=month, day = day, tzinfo = zoneinfo.ZoneInfo(iata_airports[airport]['tz']))
    return dt.astimezone(timezone.utc)

In [10]:
#the next two lines I've commented out should no longer be necessary since Joao added a more correct UTC_CRSDepTime column in the data
#df['UTC_CRSDepTime'] = df.apply(lambda row: convert_date(row['Year'], row['Month'], row['DayofMonth'], row['CRSDepTime'], row['Origin']), axis=1)
#df = df.sort_values(['Tail_Number', 'UTC_CRSDepTime'])
df['UTC_CRSDepTime'] = pd.to_datetime(df['UTC_CRSDepTime'])
df = df.sort_values(by=['Tail_Number', 'UTC_CRSDepTime']).reset_index(drop=True)
df['delta_t'] = (df.groupby('Tail_Number')['UTC_CRSDepTime'].diff().dt.total_seconds() / 3600).fillna(0)

In [11]:
#removing cancelled flights and any rows that have NaN for some reason
df = df[df['Cancelled'] ==0]
df = df.dropna()

In [12]:
#no longer being used because it's too slow
def convert_to_local_hour(utc_time, timezone):
    return utc_time.tz_localize('UTC').tz_convert(timezone).hour

In [13]:
#made with LLM, extracts the hour from CRSDepHour
def extract_military_hour(time_val):
    """Extract hour from military time (e.g., 18 -> 0, 1155 -> 11, 2319 -> 23)"""
    # Convert to string and strip whitespace
    time_str = str(time_val).strip()

    if len(time_str) <= 2:
        # Values like '18' or '5' are minutes-only format → hour = 0
        return 0
    elif len(time_str) == 3:
        # Values like '930' → hour = 9
        return int(time_str[0])
    elif len(time_str) == 4:
        # Values like '1155', '2319' → first two digits are the hour
        return int(time_str[:2])

In [14]:
#creates the hour column
df['Hour'] = df['CRSDepTime'].apply(extract_military_hour) #df.apply(
    #lambda row: convert_to_local_hour(row['UTC_CRSDepTime'], zoneinfo.ZoneInfo(iata_airports[row['Origin']]['tz'])), axis=1
#)

In [15]:
#cyclic hour categories
df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

In [17]:
#TargetEncoder() was really slow so this (LLM-generated) manually target encodes
def target_encode(train_df, val_df, cat_cols, target_col):
    """
    Manually compute target encoding using pandas groupby mean.
    Much faster than sklearn TargetEncoder for large dataframes.
    """
    encoding_maps = {}

    for col in cat_cols:
        # Compute mean target per category on train only
        means = train_df.groupby(col)[target_col].mean()
        global_mean = train_df[target_col].mean()  # fallback for unseen categories

        encoding_maps[col] = means

        # Apply to train and val
        train_df[col] = train_df[col].map(means).fillna(global_mean)
        val_df[col]   = val_df[col].map(means).fillna(global_mean)

    return train_df, val_df, encoding_maps


In [18]:
#splits up to make a training and validation set. should be adjusted based on whatever time period you're looking at.
train_df = df[(df['Month'] < 8)].copy()
val_df = df[(df['Month'] > 7)].copy()

#fields that you want to TargetEncode
cat_cols_to_encode = ['Origin', 'Dest', 'Reporting_Airline']

train_df, val_df, encoding_maps = target_encode(train_df, val_df, cat_cols_to_encode, 'DepDelayMinutes')

In [19]:
#frees memory ideally
del df

In [21]:
#returns sequences of a certain length, and the next flight in the sequence, and that next flight's label which here is DepDelayMinutes
def build_sequences(data, sequence_length):
    X_cont_list = []
    X_target_cont_list = []
    y_list = []

    for tail, group in data.groupby('Tail_Number'):
        group = group.sort_values('CRSDepTime').reset_index(drop=True)

        cont_values = group[cont_features].to_numpy(dtype=float)
        target_cont_values = group[target_cont_features].to_numpy(dtype=float)
        labels = group['DepDelayMinutes'].to_numpy()

        for i in range(len(group) - sequence_length):
            X_cont_list.append(cont_values[i : i + sequence_length])
            X_target_cont_list.append(target_cont_values[i+sequence_length])

            y_list.append(labels[i + sequence_length])

    X_cont = np.array(X_cont_list)
    X_target_cont = np.array(X_target_cont_list)
    y = np.array(y_list)

    return X_cont, X_target_cont, y


In [22]:
#runs the model, you can modify sequence_length here.
X_cont_train, X_target_cont_train, y_train = build_sequences(train_df, sequence_length=3)
X_cont_val, X_target_cont_val, y_val = build_sequences(val_df, sequence_length=3)

train_dataset = FlightDataset(X_cont_train, X_target_cont_train, y_train)
val_dataset = FlightDataset(X_cont_val,X_target_cont_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset,   batch_size=256, shuffle=False)

model = FlightDelayRNN(
    n_cont_features = len(cont_features),
    n_target_cont_features = len(target_cont_features)
).to(device)

train_model(model, train_loader, val_loader)

Beginning Epoch 1
Epoch   1 | Train Loss: 3285.3671 | Val Loss: 2724.5160
Beginning Epoch 2
Epoch   2 | Train Loss: 3266.1602 | Val Loss: 2748.1349
Beginning Epoch 3
Epoch   3 | Train Loss: 3262.1140 | Val Loss: 2727.3298
Beginning Epoch 4
Epoch   4 | Train Loss: 3261.7146 | Val Loss: 2728.9747
Training complete. Best model saved.


In [ ]:
#this function was mostly written by LLM
def evaluate_model(model, test_loader):
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    model.load_state_dict(torch.load('best_model.pt'))
    model.eval()

    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for x_cont, x_target_cont, labels in test_loader:
            x_cont        = x_cont.to(device)
            x_target_cont = x_target_cont.to(device)

            preds = model(x_cont, x_target_cont)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    mse  = mean_squared_error(all_labels,  all_preds)
    mae  = mean_absolute_error(all_labels, all_preds)
    r2   = r2_score(all_labels,            all_preds)

    print(f'MSE:  {mse:.4f}')
    print(f'RMSE: {np.sqrt(mse):.4f}')   # in same units as delay (minutes)
    print(f'MAE:  {mae:.4f}')            # average error in minutes
    print(f'R2:   {r2:.4f}')             # 1.0 = perfect, 0.0 = baseline

    return all_preds, all_labels

In [ ]:
evaluate_model(model, val_loader)

MSE:  1850.1150
RMSE: 43.0130
MAE:  17.3417
R2:   0.0101


(array([20.279991, 22.071293, 21.540392, ..., 23.036823, 21.631002,
        23.013996], dtype=float32),
 array([ 0., 67.,  0., ...,  0., 22., 22.], dtype=float32))

Below is code for linear regression as a baseline.

In [25]:
ct = ColumnTransformer(
    [
    ('onehot', OneHotEncoder(handle_unknown='ignore'), ['Tail_Number'])
    ], remainder = 'passthrough'
)

linear_reg = Pipeline([
    ('columntransform', ct),
    ('model', LinearRegression())
])

In [26]:
target = ['DepDelayMinutes']
features = ['Origin', 'Dest', 'Reporting_Airline', 'Tail_Number', 'Hour_sin','Hour_cos', 'Month_sin','Month_cos', 'DayofMonth', 'delta_t', 'Year', 'CRSDepTime', 'CRSArrTime', 'dep_mx2t', 'dep_mn2t', 'dep_mxtpr', 'dep_fg10', 'dep_u10', 'dep_v10',
                 'arr_mx2t', 'arr_mn2t', 'arr_mxtpr', 'arr_fg10', 'arr_u10', 'arr_v10']

In [27]:
start_fit_lin = time.perf_counter()
linear_reg.fit(train_df[features], train_df[target])
end_fit_lin = time.perf_counter()
time_fit_lin = end_fit_lin - start_fit_lin
print(f'fit took {time_fit_lin}')

start_pred_lin = time.perf_counter()
lin_pred = linear_reg.predict(val_df[features])
end_pred_lin = time.perf_counter()
time_pred_lin = end_pred_lin - start_pred_lin
print(f'pred took {time_pred_lin}')

print(rmse(val_df[target], lin_pred))
print(mean_absolute_error(val_df[target], lin_pred))
print(r2_score(val_df[target], lin_pred))

fit took 16.696119520999673
pred took 1.8878285939999842
53.440742103997756
28.293658787903535
-0.036512722902188655
